In [2]:
import sys
import os
import pandas as pd

sys.path.append(os.path.abspath('..'))

from src.data_loader import load_all_tables
from src.cleaning import (
    clean_orders,
    clean_geolocation,
    clean_products,
    aggregate_payments,
    clean_reviews,
    clean_strings
)
from src.features import build_master_dataset

In [3]:
# 1. Load the raw data
raw_data_path = '../data/raw'
tables = load_all_tables(raw_data_path)

print("\n--- Starting Data Cleaning ---")

# 2. Clean Orders
df_orders_delivered, df_orders_all = clean_orders(tables['olist_orders_dataset'])
print(f"Orders cleaned. Delivered only: {len(df_orders_delivered):,} rows.")

# 3. Clean Geolocation (dedupe + collapse to 1 row per zip prefix)
# NOTE: not merged into the master order-level dataset below — this is cleaned
# here so it's ready for whichever notebook/SQL query needs zip-level lat/lng
# or city/state (e.g. state delivery performance). Confirm this matches how
# it's actually consumed downstream before relying on it.
df_geo_clean = clean_geolocation(tables['olist_geolocation_dataset'])
print(f"Geolocation cleaned: {len(tables['olist_geolocation_dataset']):,} raw rows -> {len(df_geo_clean):,} unique zip prefixes.")

# 4. Clean Products (Merge Translation)
df_products_clean = clean_products(tables['olist_products_dataset'], tables['product_category_name_translation'])
print(f"Products cleaned and translated. Categories filled: {df_products_clean['product_category_name_english'].isnull().sum()} nulls.")

# 5. Aggregate Payments
df_payments_clean = aggregate_payments(tables['olist_order_payments_dataset'])
print(f"Payments aggregated: {len(df_payments_clean):,} unique orders.")

# 6. Clean Reviews (dedup on review_id, then collapsed to 1 row per order_id
# so the merge in build_master_dataset can't fan out order rows)
df_reviews_clean = clean_reviews(tables['olist_order_reviews_dataset'])
print(f"Reviews cleaned: {len(df_reviews_clean):,} unique reviews (1 per order).")

# 7. String & State Cleaning for Dimension Tables
df_customers_clean = clean_strings(tables['olist_customers_dataset'], state_cols=['customer_state'])
df_sellers_clean = clean_strings(tables['olist_sellers_dataset'], state_cols=['seller_state'])
print("String standardization and state code uppercase formatting complete.")

LOADING RAW DATA
✓ olist_customers_dataset                           99,441 rows ×   5 columns
✓ olist_geolocation_dataset                      1,000,163 rows ×   5 columns
✓ olist_order_items_dataset                        112,650 rows ×   7 columns
✓ olist_order_payments_dataset                     103,886 rows ×   5 columns
✓ olist_order_reviews_dataset                       99,224 rows ×   7 columns
✓ olist_orders_dataset                              99,441 rows ×   8 columns
✓ olist_products_dataset                            32,951 rows ×   9 columns
✓ olist_sellers_dataset                              3,095 rows ×   4 columns
✓ product_category_name_translation                     71 rows ×   2 columns
Successfully loaded 9 tables

--- Starting Data Cleaning ---
Orders cleaned. Delivered only: 96,470 rows.
Geolocation cleaned: 1,000,163 raw rows -> 19,015 unique zip prefixes.
Products cleaned and translated. Categories filled: 0 nulls.
Payments aggregated: 99,440 unique orders.


In [4]:
# Create the before/after report as requested in the spec
report_dir = '../outputs/ai_insights/reports'
os.makedirs(report_dir, exist_ok=True)
report_path = os.path.join(report_dir, 'cleaning_summary_report.txt')

with open(report_path, 'w', encoding='utf-8') as f:
    f.write("========== DATA CLEANING SUMMARY ==========\n\n")

    f.write("--- ORDERS TABLE ---\n")
    f.write(f"Raw Rows: {len(tables['olist_orders_dataset']):,}\n")
    f.write(f"Cleaned (All Statuses): {len(df_orders_all):,}\n")
    f.write(f"Cleaned (Delivered Only, non-null delivery date): {len(df_orders_delivered):,}\n\n")

    f.write("--- GEOLOCATION TABLE ---\n")
    f.write(f"Raw Rows: {len(tables['olist_geolocation_dataset']):,}\n")
    f.write(f"Cleaned (1 row per zip prefix): {len(df_geo_clean):,}\n\n")

    f.write("--- PRODUCTS TABLE ---\n")
    f.write(f"Raw Rows: {len(tables['olist_products_dataset']):,}\n")
    f.write(f"Cleaned (missing content/dimensions dropped): {len(df_products_clean):,}\n\n")

    f.write("--- PAYMENTS TABLE ---\n")
    f.write(f"Raw Rows (Multiple per order): {len(tables['olist_order_payments_dataset']):,}\n")
    f.write(f"Cleaned (Aggregated to 1 per order): {len(df_payments_clean):,}\n\n")

    f.write("--- REVIEWS TABLE ---\n")
    f.write(f"Raw Rows: {len(tables['olist_order_reviews_dataset']):,}\n")
    f.write(f"Cleaned (1 row per order): {len(df_reviews_clean):,}\n")

print(f"Cleaning summary saved to {report_path}")

Cleaning summary saved to ../outputs/ai_insights/reports\cleaning_summary_report.txt


In [5]:
# Build the master dataset with ALL required tables passed in
df_master = build_master_dataset(
    df_orders_delivered,
    df_payments_clean,
    df_reviews_clean,
    tables['olist_order_items_dataset'],
    df_products_clean,
    df_customers_clean
)

In [6]:
print(f"Master Dataset created with {len(df_master):,} rows and {len(df_master.columns)} columns.")

# Sanity check: a left-merge chain starting from df_orders_delivered must never
# produce more rows than df_orders_delivered itself. If this ever fails, one of
# the merged tables (payments/reviews/items) isn't 1-row-per-order anymore.
assert len(df_master) == len(df_orders_delivered), (
    f"Row count mismatch: master has {len(df_master):,} rows, "
    f"expected {len(df_orders_delivered):,}. Check for a fan-out in one of the merges."
)
print("Row count check passed: no fan-out from merges.")

os.makedirs('../data/processed', exist_ok=True)
df_master.to_csv('../data/processed/master_orders.csv', index=False)

print("Saved successfully to data/processed/master_orders.csv")
print(df_master.columns.tolist())

Master Dataset created with 96,470 rows and 25 columns.
Row count check passed: no fan-out from merges.
Saved successfully to data/processed/master_orders.csv
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'is_delivered', 'delivery_days', 'delay_days', 'is_late', 'delay_bucket', 'order_month', 'order_year', 'order_hour', 'order_dayofweek', 'order_yearmonth', 'revenue_per_order', 'primary_payment_type', 'max_installments', 'review_score', 'customer_unique_id', 'customer_state', 'product_category_name_english']


In [8]:
%pip install pyodbc

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
%run ../src/sql_loader.py

--- Starting SQL Server Data Load ---
Loading tables...
LOADING RAW DATA
✓ olist_customers_dataset                           99,441 rows ×   5 columns
✓ olist_geolocation_dataset                      1,000,163 rows ×   5 columns
✓ olist_order_items_dataset                        112,650 rows ×   7 columns
✓ olist_order_payments_dataset                     103,886 rows ×   5 columns
✓ olist_order_reviews_dataset                       99,224 rows ×   7 columns
✓ olist_orders_dataset                              99,441 rows ×   8 columns
✓ olist_products_dataset                            32,951 rows ×   9 columns
✓ olist_sellers_dataset                              3,095 rows ×   4 columns
✓ product_category_name_translation                     71 rows ×   2 columns
Successfully loaded 9 tables

Loading Master Orders (Fact Table)...
Preparing Dimension Tables...
Pushing fact_orders (96,470 rows) to SQL Server...


InterfaceError: (pyodbc.InterfaceError) ('28000', '[28000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Login failed for user \'DESKTOP-A39MPDE\\DELL\'. (18456) (SQLDriverConnect); [28000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Cannot open database "olist_db" requested by the login. The login failed. (4060); [28000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Login failed for user \'DESKTOP-A39MPDE\\DELL\'. (18456); [28000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Cannot open database "olist_db" requested by the login. The login failed. (4060)')
(Background on this error at: https://sqlalche.me/e/20/rvf5)

In [13]:
import pyodbc

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    r"SERVER=DESKTOP-A39MPDE\SQLEXPRESS;"
    "DATABASE=olist_db;"
    "Trusted_Connection=yes;"
)

print("✅ Connected to olist_db successfully!")

conn.close()

✅ Connected to olist_db successfully!


In [ ]:
%run ../src/sql_loader.py

--- Starting SQL Server Data Load ---
Loading tables...
LOADING RAW DATA
✓ olist_customers_dataset                           99,441 rows ×   5 columns
✓ olist_geolocation_dataset                      1,000,163 rows ×   5 columns
✓ olist_order_items_dataset                        112,650 rows ×   7 columns
✓ olist_order_payments_dataset                     103,886 rows ×   5 columns
✓ olist_order_reviews_dataset                       99,224 rows ×   7 columns
✓ olist_orders_dataset                              99,441 rows ×   8 columns
✓ olist_products_dataset                            32,951 rows ×   9 columns
✓ olist_sellers_dataset                              3,095 rows ×   4 columns
✓ product_category_name_translation                     71 rows ×   2 columns
Successfully loaded 9 tables

Loading Master Orders (Fact Table)...
Preparing Dimension Tables...
Pushing fact_orders (96,470 rows) to SQL Server...
